# Gemma-3 4B Fine-tune (Colab)
Unsloth + Llama Factory • RAG MDN

In [ ]:
%pip install -q transformers accelerate peft datasets sentencepiece bitsandbytes unsloth llama-factory

In [ ]:
from google.colab import files
uploaded = files.upload()
import json, os
local_path = list(uploaded.keys())[0]
dataset_path = '/content/' + local_path

In [ ]:
def chunk_text(input, max_len=1500):
    parts = input.split('

')
    chunks = []
    buf = ''
    for p in parts:
        seg = p.strip()
        if not seg:
            continue
        if len(buf) + len(seg) + (2 if buf else 0) <= max_len:
            buf = (buf + ('\n\n' if buf else '') + seg)
            continue
        if buf:
            chunks.append(buf)
            buf = ''
        if len(seg) <= max_len:
            chunks.append(seg)
            continue
        s = seg
        while len(s) > max_len:
            cut = s.rfind('.', 0, max_len)
            idx = cut + 1 if cut > 200 else max_len
            chunks.append(s[:idx].strip())
            s = s[idx:].strip()
        if s:
            chunks.append(s)
    if buf:
        chunks.append(buf)
    return chunks

In [ ]:
out_path = '/content/finetune_instructions.jsonl'
items = json.load(open(dataset_path, 'r', encoding='utf-8'))
f = open(out_path, 'w', encoding='utf-8')
for it in items:
    topic = it.get('topic') or ''
    summary = it.get('summary') or ''
    detail = it.get('detailed_knowledge') or ''
    content = ('\n\n').join([x for x in [summary, detail] if x])
    for c in chunk_text(content, 1500):
        rec = { 'instruction': 'Giải thích chi tiết: ' + topic, 'input': '', 'output': c }
        f.write(json.dumps(rec, ensure_ascii=False) + '
')
f.close()
print(out_path)

## Option A: Llama Factory QLoRA

In [ ]:
!python -m llamafactory.cli train --model_name_or_path unsloth/gemma-3-4b-it-unsloth-bnb-4bit --finetuning_type lora --data_path /content/finetune_instructions.jsonl --dataset_format jsonl --template default --cutoff_len 2048 --quantization_bit 4 --lora_target q_proj,v_proj,k_proj,o_proj --per_device_train_batch_size 1 --gradient_accumulation_steps 16 --learning_rate 2e-4 --num_train_epochs 2 --save_steps 200 --logging_steps 50 --output_dir /content/outputs/gemma3-4b-mdn-lora

## Option B: Unsloth Programmatic QLoRA

In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
model_id = 'unsloth/gemma-3-4b-it-unsloth-bnb-4bit'
model, tokenizer = FastLanguageModel.from_pretrained(model_name=model_id, max_seq_length=2048, dtype=None, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=64, target_modules=['q_proj','k_proj','v_proj','o_proj'], lora_alpha=16, lora_dropout=0.0, bias='none', use_gradient_checkpointing=True, random_state=3407, use_rslora=True, loftq_config=None)
def format_example(ex):
    inst = ex['instruction']
    inp = ex.get('input','')
    out = ex['output']
    if inp:
        text = '
'.join([
            '### Instruction:', inst,
            '', '### Input:', inp,
            '', '### Response:', out
        ])
    else:
        text = '
'.join([
            '### Instruction:', inst,
            '', '### Response:', out
        ])
    return { 'text': text }
dataset = load_dataset('json', data_files='/content/finetune_instructions.jsonl', split='train')
dataset = dataset.map(format_example, remove_columns=dataset.column_names)
dc = DataCollatorForLanguageModeling(tokenizer, mlm=False)
args = TrainingArguments(output_dir='/content/outputs/gemma3-4b-mdn-lora', per_device_train_batch_size=1, gradient_accumulation_steps=16, num_train_epochs=2, learning_rate=2e-4, logging_steps=50, save_steps=200, save_total_limit=2, bf16=True if torch.cuda.is_available() else False)
def tokenize(ex):
    return tokenizer(ex['text'], truncation=True, padding='max_length', max_length=2048)
tokenized = dataset.map(tokenize, batched=True, remove_columns=['text'])
trainer = Trainer(model=model, args=args, train_dataset=tokenized, data_collator=dc)
trainer.train()
trainer.save_model('/content/outputs/gemma3-4b-mdn-lora')
tokenizer.save_pretrained('/content/outputs/gemma3-4b-mdn-lora')

## Inference Test

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
base_id = 'unsloth/gemma-3-4b-it-unsloth-bnb-4bit'
tok = AutoTokenizer.from_pretrained(base_id)
base = AutoModelForCausalLM.from_pretrained(base_id, load_in_4bit=True, device_map='auto')
lora = PeftModel.from_pretrained(base, '/content/outputs/gemma3-4b-mdn-lora')
prompt = 'Giải thích cách thêm JavaScript vào HTML'
inputs = tok(prompt, return_tensors='pt').to(lora.device)
out = lora.generate(**inputs, max_new_tokens=256)
print(tok.decode(out[0], skip_special_tokens=True))

## Save to Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r /content/outputs/gemma3-4b-mdn-lora /content/drive/MyDrive/gemma3-4b-mdn-lora